# SOAP 相似性核

**适用环境：Windows / macOS / Linux（Python 与 Jupyter）。**

## 输入与依赖

准备包含 Ge、Te 原子的三份轨迹：`split_100.xyz`、`split_300.xyz` 和 `split_800.xyz`，放在同一目录。输入应能由 ASE 读取，原子坐标使用 Å。`split_100.xyz` 用于构造参考描述符，另外两份用于绘图比较；请使用自己的轨迹，并核对温度标签与实际数据对应。

运行前，将环境变量 `SOAP_DATA_DIR` 设为该目录的实际路径；也可在 Jupyter 中用 `%env SOAP_DATA_DIR=实际路径` 设置。Python 环境需有 ASE、NumPy、DScribe、Matplotlib 和 SciencePlots。下面保留原有 SOAP 参数、参考平均方式和采样间隔。


In [ ]:
from dscribe.descriptors import SOAP
species = ["Ge", "Te"]
r_cut = 3.0
sigma = 0.3
n_max = 8
l_max = 8
soap = SOAP(species=species, r_cut=r_cut, sigma=sigma, n_max=n_max, l_max=l_max)
soap.get_number_of_features()

In [ ]:
from ase.io import read
import numpy as np
import os
from pathlib import Path

input_root = os.environ.get("SOAP_DATA_DIR")
if not input_root:
    raise ValueError("请先设置 SOAP_DATA_DIR，指向包含三份 split_*.xyz 轨迹的目录。")
data_dir = Path(input_root).expanduser().resolve()
required_names = ("split_100.xyz", "split_300.xyz", "split_800.xyz")
missing = [name for name in required_names if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"{data_dir} 缺少轨迹文件：{', '.join(missing)}")

xyz_files = read(data_dir / "split_100.xyz", index=':')
soap_vectors = []
for atoms in xyz_files:
    Ge_indices = [atom.index for atom in atoms if atom.symbol == 'Ge']
    soap_vector = soap.create(atoms, centers=Ge_indices)
    soap_vector_avg = np.mean(soap_vector, axis=0)
    soap_vectors.append(soap_vector_avg)

p_j = np.mean(soap_vectors, axis=0).reshape(1, -1)
p_j_norm = p_j / np.linalg.norm(p_j)

In [ ]:
from ase.io import read
import matplotlib.pyplot as plt
import numpy as np
import scienceplots
plt.style.use(['science','ieee','no-latex','bright'])

def _plot(ax, filename, p_j_norm, label):
    xyz_files = read(filename, index='::100')
    kj_list = []
    for atoms in xyz_files:
        Ge_indices = [atom.index for atom in atoms if atom.symbol == 'Ge']
        for i in Ge_indices:
            p_i = soap.create(atoms, centers=[i])
            p_i_norm = p_i / np.linalg.norm(p_i)
            k_j = np.dot(p_i_norm, p_j_norm.T) ** 2
            kj_list.append(k_j[0][0])

    bins = np.linspace(0, 1, 1001)
    hist, bin_edges = np.histogram(kj_list, bins=bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    plt.plot(bin_centers, hist, linestyle='-', label=label)

plt.figure(figsize=(4, 3))
ax1 = plt.gca()
_plot(ax1, data_dir / "split_300.xyz", p_j_norm, label="300K")
_plot(ax1, data_dir / "split_800.xyz", p_j_norm, label="800K")
plt.xlabel(r'$k_j$ Value')
plt.ylabel('Distribution')
plt.xlim(0.8, 1)
plt.ylim(0, None)
plt.yticks([])
plt.legend()
plt.show()